# Train Jack SaaS character LoRA (SDXL)

**Runtime:** T4 GPU (free Colab tier is sufficient — ~3–5 hours).

**Before running:**
1. Set `Runtime → Change runtime type → T4 GPU`.
2. In `🔑 Secrets` (left sidebar) add:
   - `HF_TOKEN` — HuggingFace token with write scope.
   - `HF_USERNAME` — your HF username (e.g. `ferdinandb`).
   - `HF_DATASET` — full dataset name from `npm run hf-upload-dataset` (e.g. `ferdinandb/jack-saas-training-v1`).
3. Click `Runtime → Run all`.

The notebook will: download your training set → train an SDXL LoRA via kohya_ss → upload the resulting `.safetensors` back to HuggingFace as a new private model repo. The final cell prints the LoRA URL you'll paste into `.env` as `LORA_URL`.

## (Optional) Mount Google Drive

Useful if you want checkpoints to survive a runtime disconnect. Skip if you don't care.

In [ ]:
MOUNT_DRIVE = False  # flip to True if you want /content/drive persistence
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR_BASE = '/content/drive/MyDrive/jack-lora'
else:
    OUTPUT_DIR_BASE = '/content/output'
import os
os.makedirs(OUTPUT_DIR_BASE, exist_ok=True)
print('Output dir:', OUTPUT_DIR_BASE)

## 1. Install dependencies

Clones kohya_ss/sd-scripts and pins compatible versions of accelerate / diffusers / peft / bitsandbytes.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/kohya-ss/sd-scripts.git || (cd sd-scripts && git pull)
%cd /content/sd-scripts
!pip install -q --upgrade pip
!pip install -q -r requirements.txt
!pip install -q xformers==0.0.27 bitsandbytes==0.43.1 accelerate==0.30.1 diffusers==0.27.2 peft==0.10.0 huggingface_hub==0.23.0
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — change runtime to T4 GPU')

## 2. Read Colab secrets

Fails fast if secrets aren't set — saves you 4 hours of training before discovering you can't upload the result.

In [ ]:
from google.colab import userdata

def _get(name):
    try:
        v = userdata.get(name)
    except Exception as e:
        raise RuntimeError(f'Colab secret {name!r} not set. Open the 🔑 panel and add it.') from e
    if not v:
        raise RuntimeError(f'Colab secret {name!r} is empty.')
    return v

HF_TOKEN = _get('HF_TOKEN')
HF_USERNAME = _get('HF_USERNAME')
HF_DATASET = _get('HF_DATASET')   # e.g. "ferdinandb/jack-saas-training-v1"

print('User:    ', HF_USERNAME)
print('Dataset: ', HF_DATASET)

## 3. Download training dataset from HuggingFace

Pulls `jack-training.zip` from the private dataset repo and extracts it into kohya_ss's expected layout: `/content/training_data/10_jacksaas/{image, caption}` pairs.

In [ ]:
import os, zipfile, shutil
from huggingface_hub import hf_hub_download, login

login(token=HF_TOKEN, add_to_git_credential=False)

TRIGGER = 'jacksaas'
REPEATS = 10  # kohya_ss reads {N}_{class} folder names as N repeats per epoch
TRAIN_DIR = '/content/training_data'
CLASS_DIR = f'{TRAIN_DIR}/{REPEATS}_{TRIGGER}'

# Clean slate
if os.path.exists(TRAIN_DIR):
    shutil.rmtree(TRAIN_DIR)
os.makedirs(CLASS_DIR, exist_ok=True)

zip_path = hf_hub_download(
    repo_id=HF_DATASET,
    repo_type='dataset',
    filename='jack-training.zip',
    token=HF_TOKEN,
)
print('Downloaded zip:', zip_path)

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(CLASS_DIR)

imgs = [f for f in os.listdir(CLASS_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]
txts = [f for f in os.listdir(CLASS_DIR) if f.lower().endswith('.txt')]
print(f'Extracted: {len(imgs)} images, {len(txts)} captions')
assert len(imgs) >= 5, 'Too few images — re-check prepare-training step.'

## 4. Training configuration

SDXL LoRA hyperparameters tuned for T4 (16 GB VRAM). Adjust `MAX_TRAIN_STEPS` first if you want to iterate faster.

In [ ]:
BASE_MODEL    = 'stabilityai/stable-diffusion-xl-base-1.0'
RESOLUTION    = 1024
BATCH_SIZE    = 1
GRAD_ACCUM    = 2
NETWORK_DIM   = 16
NETWORK_ALPHA = 16
LEARNING_RATE = 1e-4
MAX_TRAIN_STEPS  = 1500
SAVE_EVERY_STEPS = 250
MIXED_PRECISION  = 'fp16'
OPTIMIZER        = 'AdamW8bit'
OUTPUT_NAME      = 'jack-lora'
OUTPUT_DIR       = OUTPUT_DIR_BASE
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Will train', MAX_TRAIN_STEPS, 'steps at', RESOLUTION, 'with rank', NETWORK_DIM)

## 5. Launch training

Expect ~3–5 hours on T4. Output checkpoints land in `OUTPUT_DIR/jack-lora-step000NNNN.safetensors` plus a final `jack-lora.safetensors`.

In [ ]:
%cd /content/sd-scripts

cmd = f'''accelerate launch --num_cpu_threads_per_process 1 sdxl_train_network.py \
  --pretrained_model_name_or_path={BASE_MODEL} \
  --train_data_dir={TRAIN_DIR} \
  --resolution={RESOLUTION},{RESOLUTION} \
  --output_dir={OUTPUT_DIR} \
  --output_name={OUTPUT_NAME} \
  --save_model_as=safetensors \
  --network_module=networks.lora \
  --network_dim={NETWORK_DIM} \
  --network_alpha={NETWORK_ALPHA} \
  --train_batch_size={BATCH_SIZE} \
  --gradient_accumulation_steps={GRAD_ACCUM} \
  --learning_rate={LEARNING_RATE} \
  --optimizer_type={OPTIMIZER} \
  --lr_scheduler=cosine_with_restarts \
  --lr_warmup_steps=100 \
  --max_train_steps={MAX_TRAIN_STEPS} \
  --save_every_n_steps={SAVE_EVERY_STEPS} \
  --mixed_precision={MIXED_PRECISION} \
  --save_precision={MIXED_PRECISION} \
  --xformers \
  --cache_latents \
  --gradient_checkpointing \
  --max_data_loader_n_workers=2 \
  --seed=42'''

print(cmd)
!{cmd}

## 6. Locate the final checkpoint

Picks the highest-step .safetensors. Copies it to a stable path: `/content/jack-lora.safetensors`.

In [ ]:
import glob, re, shutil

candidates = sorted(glob.glob(f'{OUTPUT_DIR}/{OUTPUT_NAME}*.safetensors'))
assert candidates, f'No .safetensors found in {OUTPUT_DIR}. Training likely failed — scroll up.'

def step_of(path):
    m = re.search(r'(?:step|-)(\d+)\.safetensors$', path)
    return int(m.group(1)) if m else 10**9   # un-suffixed final goes last
candidates.sort(key=step_of)
best = candidates[-1]

final_path = '/content/jack-lora.safetensors'
shutil.copy(best, final_path)
print('All checkpoints:')
for c in candidates: print(' ', c, f'({os.path.getsize(c)/1e6:.1f} MB)')
print('Selected:      ', best)
print('Copied to:     ', final_path)

## 7. Upload LoRA to HuggingFace

Creates a new private model repo `{HF_USERNAME}/jack-saas-lora-v{N}` (auto-increments) and uploads `jack-lora.safetensors`.

**Copy the printed `LORA_URL=…` line into your local `.env`** — that's what `npm run generate-stills-lora` reads.

In [ ]:
import re
from huggingface_hub import HfApi, create_repo, upload_file

api = HfApi(token=HF_TOKEN)

prefix = 'jack-saas-lora'
pattern = re.compile(rf'^{re.escape(HF_USERNAME)}/{re.escape(prefix)}-v(\d+)$')
highest = 0
try:
    for m in api.list_models(author=HF_USERNAME):
        match = pattern.match(m.id)
        if match:
            highest = max(highest, int(match.group(1)))
except Exception as e:
    print('WARN: could not list existing models —', e)

version = highest + 1
repo_id = f'{HF_USERNAME}/{prefix}-v{version}'
print('Target repo:', repo_id)

create_repo(repo_id=repo_id, repo_type='model', private=True, exist_ok=True, token=HF_TOKEN)
upload_file(
    path_or_fileobj=final_path,
    path_in_repo='jack-lora.safetensors',
    repo_id=repo_id,
    repo_type='model',
    token=HF_TOKEN,
)

lora_url = f'https://huggingface.co/{repo_id}/resolve/main/jack-lora.safetensors'
print('')
print('✅ DONE')
print('Repo:    ', f'https://huggingface.co/{repo_id}')
print('LoRA URL:', lora_url)
print('')
print('Paste this line into your local .env:')
print(f'  LORA_URL={lora_url}')